# 01_data_agreement_template — governance metadata and approval flow

Use this notebook to define and approve agreement-level governance metadata once, then reuse it as read-only context in downstream notebooks.


In [ ]:
%run 00_env_config


In [ ]:
from fabricops_kit import (
    build_governance_prompt_context,
    run_governance_widget,
    get_governance_widget_results,
    save_governance_widget_results,
    load_governance_context,
    prepare_governance_profile_input,
    suggest_personal_identifier_classifications,
    extract_personal_identifier_suggestions,
    write_metadata_rows,
    get_path,
)

# ---- User placeholders ----
ENV_NAME = "dev"
DATASET_NAME = "sample_agreement_dataset"
TABLE_NAME = "minimal_source"
AGREEMENT_ID = "REPLACE_AGREEMENT_ID"
APPROVED_USAGE = "REPLACE approved usage"
BUSINESS_CONTEXT = "REPLACE business context"
OWNERSHIP = "REPLACE owner/team"
PERMISSIONS = "REPLACE who can access"
RESTRICTIONS = "REPLACE restrictions"
RELATED_NOTEBOOK_LINKS = ["REPLACE notebook link"]
AGREEMENT_CONTEXT = "REPLACE agreement context summary"
STEWARD_NOTES = "REPLACE steward notes"
SAVE_TO_METADATA = True


In [ ]:
governance_prompt_context = build_governance_prompt_context(
    business_context=BUSINESS_CONTEXT,
    approved_usage=APPROVED_USAGE,
    dataset_context=AGREEMENT_CONTEXT,
    steward_notes=STEWARD_NOTES,
)
agreement_context = {
    "agreement_id": AGREEMENT_ID,
    "approved_usage": APPROVED_USAGE,
    "business_context": BUSINESS_CONTEXT,
    "ownership": OWNERSHIP,
    "permissions": PERMISSIONS,
    "restrictions": RESTRICTIONS,
    "classification": "approved_by_human_review",
    "sensitivity": "approved_by_human_review",
    "pii": "approved_by_human_review",
    "related_notebook_links": RELATED_NOTEBOOK_LINKS,
    "agreement_context": AGREEMENT_CONTEXT,
}
agreement_context


## Optional AI-assisted governance suggestions

If a profile table and approved business context already exist, use AI suggestions; otherwise start with manual seed suggestions.


In [ ]:
# Optional: use profile + approved business context when available.
# Example source table names can be replaced by your metadata table names.
use_ai_profile_input = False
if use_ai_profile_input:
    profile_rows = [r.asDict(recursive=True) for r in spark.table("METADATA_PROFILE_ROWS").filter(f"table_name = '{TABLE_NAME}'").collect()]
    approved_business_context_rows = [r.asDict(recursive=True) for r in spark.table("METADATA_COLUMN_CONTEXT").filter("status = 'approved'").collect()]
    prepared_rows = prepare_governance_profile_input(profile_rows, table_name=TABLE_NAME, column_contexts=approved_business_context_rows)
    prepared_df = spark.createDataFrame(prepared_rows)
    governance_response_df = suggest_personal_identifier_classifications(
        prepared_profile_df=prepared_df,
        prompt=CONFIG.ai_prompt_config.governance_personal_identifier_template,
    )
    governance_suggestions = extract_personal_identifier_suggestions(governance_response_df)
else:
    governance_suggestions = [
        {"column_name": "customer_id", "approved_business_context": "Business identifier for customer records", "ai_suggested_personal_identifier_classification": "indirect_identifier", "confidentiality_label": "confidential"},
        {"column_name": "email", "approved_business_context": "Customer communication contact", "ai_suggested_personal_identifier_classification": "direct_identifier", "confidentiality_label": "restricted"},
    ]

governance_suggestions


In [ ]:
metadata_path = get_path(ENV_NAME, "metadata", config=FABRIC_CONFIG)
run_governance_widget(
    suggestions=governance_suggestions,
    environment_name=ENV_NAME,
    dataset_name=DATASET_NAME,
    table_name=TABLE_NAME,
)
print("Widget launched. Complete approvals/rejections, then run the next cell to collect/save results.")


In [ ]:
widget_results = get_governance_widget_results(agreement_context=agreement_context)
approved_rows = widget_results["approved_rows"]
rejected_rows = widget_results["rejected_rows"]
print(f"Approved rows: {len(approved_rows)} | Rejected rows: {len(rejected_rows)}")
if SAVE_TO_METADATA:
    saved_rows = save_governance_widget_results(
        spark,
        metadata_path=metadata_path,
        agreement_context=agreement_context,
        metadata_table_name="METADATA_COLUMN_GOVERNANCE",
        save_mode="append",
    )
    print(f"Saved approved governance rows: {len(saved_rows)}")


In [ ]:
# Optional agreement-level snapshot row for METADATA_DATA_AGREEMENT.
# This appends one row per run; implement deterministic merge/upsert in project pipelines if strict idempotency is required.
agreement_record = {
    "agreement_id": AGREEMENT_ID,
    "dataset_name": DATASET_NAME,
    "table_name": TABLE_NAME,
    **agreement_context,
}
if SAVE_TO_METADATA:
    write_metadata_rows(
        spark,
        rows=[agreement_record],
        metadata_path=metadata_path,
        table_name="METADATA_DATA_AGREEMENT",
        mode="append",
    )
agreement_record


In [ ]:
# Downstream read-only context load (exploration/pipeline notebooks).
approved_governance_df = spark.table("METADATA_COLUMN_GOVERNANCE")
agreement_df = spark.table("METADATA_DATA_AGREEMENT")
read_only_context = load_governance_context(
    approved_governance_df,
    agreement_rows=agreement_df,
    agreement_id=AGREEMENT_ID,
    dataset_name=DATASET_NAME,
    table_name=TABLE_NAME,
)
read_only_context


## Re-run guidance

- Keep `AGREEMENT_ID` stable for the same agreement.
- Re-run safely by approving only changed rows; downstream notebooks should always load `status='approved'` context via `load_governance_context(...)`.
- No DQ enforcement changes are made in this notebook.
